# Result analysis

Loads JSONL rows from `results/` and plots:

1. TTFT vs context length
2. Decode throughput (tokens/sec) vs context length
3. Peak GPU memory vs KV-cache estimate
4. Latency vs throughput scatter

Runs in Colab or locally — needs only pandas + matplotlib.

## 1. Get the code (skip if running locally inside the repo)

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/LLM_Inference.git"  # <-- EDIT IF CLONING

if not os.path.exists("results"):
    !git clone {REPO_URL}
    repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    %cd {repo_name}

!ls results/

## 2. Load every JSONL row into a DataFrame

In [ ]:
import json
from pathlib import Path
import pandas as pd

rows = []
for path in sorted(Path("results").glob("*.jsonl")):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} rows")
df.head()

## 3. Quick overview

In [ ]:
print(f"Successes: {df['success'].sum()} / {len(df)}")
print(f"Models:    {df['model_name'].unique().tolist()}")
print(f"Backends:  {df['backend'].unique().tolist()}")
print(f"Hardware:  {df['hardware'].unique().tolist()}")
print(f"Contexts:  {sorted(df['context_length'].unique().tolist())}")

## 4. TTFT vs context length

Linear-ish growth is the expected baseline. Steeper growth suggests memory pressure or prefill recomputation.

In [ ]:
import matplotlib.pyplot as plt

ok = df[df["success"]].sort_values("context_length")
fig, ax = plt.subplots(figsize=(8, 5))
for (model, backend), g in ok.groupby(["model_name", "backend"]):
    ax.plot(g["context_length"], g["ttft_seconds"], marker="o", label=f"{model} ({backend})")
ax.set_xscale("log", base=2)
ax.set_xlabel("Context length (tokens)")
ax.set_ylabel("TTFT (s)")
ax.set_title("Time to first token vs context length")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Decode throughput vs context length

Tokens/sec at decode is typically flat or mildly degrading with context, because every decode step reads the whole KV cache.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for (model, backend), g in ok.groupby(["model_name", "backend"]):
    ax.plot(g["context_length"], g["tokens_per_second"], marker="o", label=f"{model} ({backend})")
ax.set_xscale("log", base=2)
ax.set_xlabel("Context length (tokens)")
ax.set_ylabel("Tokens / sec")
ax.set_title("Decode throughput vs context length")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Memory: measured peak vs KV-cache estimate

The gap between the two is everything else (model weights, activations, framework overhead). When they grow together, KV cache is the dominant cost — which is the whole motivation for paged-KV designs (vLLM).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for (model, backend), g in ok.groupby(["model_name", "backend"]):
    if g["peak_gpu_memory_gb"].notna().any():
        ax.plot(g["context_length"], g["peak_gpu_memory_gb"],
                marker="o", label=f"{model} measured peak")
    ax.plot(g["context_length"], g["kv_cache_memory_gb"],
            marker="s", linestyle="--", label=f"{model} KV-cache estimate")
ax.set_xscale("log", base=2)
ax.set_xlabel("Context length (tokens)")
ax.set_ylabel("GB")
ax.set_title("Memory: peak GPU vs KV-cache estimate")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Latency vs throughput scatter

Each point is one (model, context length) run. Up-and-to-the-left wins. Useful for spotting Pareto-frontier candidates as more backends/models get added.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for (model, backend), g in ok.groupby(["model_name", "backend"]):
    ax.scatter(g["total_latency_seconds"], g["tokens_per_second"],
               label=f"{model} ({backend})")
    for _, row in g.iterrows():
        ax.annotate(f"{int(row['context_length'])}",
                    (row["total_latency_seconds"], row["tokens_per_second"]),
                    fontsize=8, alpha=0.7)
ax.set_xlabel("Total latency (s)")
ax.set_ylabel("Tokens / sec")
ax.set_title("Latency vs throughput (labels = context length)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## What to look at

- **Where TTFT bends sharply upward** — that's the prefill/memory cliff for this model on this hardware.
- **Where throughput drops** — decode is becoming KV-cache-bandwidth bound.
- **Failure rows** (in `results/*.jsonl` with `success: false`) mark the OOM wall. Check `summarize_results.py` output for the list.